# 1. Házi feladat – Folyamillesztés
**Generatív AI és inverz módszerek a képszintézisben** | BME VIK, 2026 tavasz

## Kötelező feladatok
1. **DDPM 2D pontokon**
   - `q_sample`
   - `p_losses`
   - `ddpm_sample_step`
2. **2D Flow Matching**
   - `fm_2d_train_step`
3. **Feltételes Flow Matching MNIST-en**
   - `fm_train_step`
   - `fm_euler_sample`

## Opcionális feladatok
- `ddim_sample_step`
- `fm_cfg_sample`

## Javasolt munkamenet
1. először csak a **kötelező** részeket csináld meg,
2. minden TODO után futtasd a közvetlen sanity check cellát,
3. csak ezután menj tovább az opcionális bónusz feladatokra,
4. először mindig a **shape**-eket ellenőrizd.


In [ ]:

# Környezet és alapbeállítások
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch torchvision matplotlib numpy scikit-learn tqdm --quiet

import math
import random
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from sklearn.datasets import make_moons
from torch.utils.data import DataLoader, TensorDataset, Subset

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Eszköz:", device)

# Rövid / hosszú futási mód
RUN_LONG_TRAINING = False

@dataclass
class CFG:
    n_points_2d: int = 4000 if not RUN_LONG_TRAINING else 8000
    batch_size_2d: int = 256
    epochs_ddpm_2d: int = 100 if not RUN_LONG_TRAINING else 500
    epochs_fm_2d: int = 100 if not RUN_LONG_TRAINING else 400
    steps_per_epoch_fm2d: int = 20 if not RUN_LONG_TRAINING else 30

    mnist_train_subset: int = 12000 if not RUN_LONG_TRAINING else 60000
    batch_size_mnist: int = 128 if not RUN_LONG_TRAINING else 256
    epochs_mnist: int = 3 if not RUN_LONG_TRAINING else 15

cfg = CFG()
print(cfg)

## Kapcsolat a korábbi PyTorch notebookokkal

Ez a házi ugyanazokat a mintákat használja,
mint a korábbi MNIST-osztályozó és denoiser notebookok, csak most az inputhoz hozzáadunk
**egy időváltozót** `t`, és a cél sem mindig ugyanaz.

| Korábbi notebook | Most a háziban |
|---|---|
| `model(x)` | `model(x_t, t)` vagy `model(x_t, t, labels)` |
| zajos kép → tiszta kép | zajos / interpolált állapot → zaj vagy sebesség |
| `loss = criterion(pred, target)` | pontosan ugyanez |
| `optimizer.zero_grad(); loss.backward(); optimizer.step()` | pontosan ugyanez |
| `model.eval(); with torch.no_grad():` | pontosan ugyanez |

## Mi az igazán új?
- **DDPM-ben** az idő `t` egy **egész index**, ezért `dtype=torch.long`.
- **Flow matchingben** az idő `t` egy **folytonos valós szám** a `[0,1]` intervallumban.
- Minden batch-elemhez más `t` tartozhat, ezért kell néha az `extract()` és a broadcasting.

## Fontos stíluselv ebben a notebookban
A **jó olvashatóság fontosabb**, mint a tömör vagy „okos” PyTorch-kód.
A cél az, hogy lásd, **milyen tensorból milyen tensor lesz**.


In [ ]:
# Kis segédfüggvények a debugoláshoz

mse_criterion = nn.MSELoss()

def print_shape(name, x):
    print(f"{name:>16}: shape={tuple(x.shape)}, dtype={x.dtype}, device={x.device}")

def show_2d_points(points, title="", ax=None, color=None, s=4, alpha=0.5):
    if isinstance(points, torch.Tensor):
        points = points.detach().cpu().numpy()
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(points[:, 0], points[:, 1], s=s, alpha=alpha, c=color)
    ax.set_title(title)
    ax.set_aspect("equal")
    return ax

def check_no_nan(name, x):
    assert torch.isfinite(x).all(), f"{name} NaN vagy inf értéket tartalmaz!"

# Gyors shape-példa
x_demo = torch.randn(4, 1, 28, 28, device=device)
t_demo = torch.rand(4, device=device)
t_demo_expand = t_demo.view(-1, 1, 1, 1)
print_shape("x_demo", x_demo)
print_shape("t_demo", t_demo)
print_shape("t_expand", t_demo_expand)


---

# 1. feladat – DDPM 2D pontfelhőn

Ebben a részben először kis 2D pontokon dolgozunk. Ez sokkal könnyebben debugolható, mint a képek.

## Matematikai háttér röviden
A forward folyamat:

$$
x_t = \sqrt{\bar\alpha_t} \, x_0 + \sqrt{1-\bar\alpha_t} \, \varepsilon,
$$
ahol $\varepsilon \propto \mathcal{N}(0,I)$ standard normális eloszlású zaj, $ \bar\alpha_t = \prod_{\tau = 1}^t \alpha_\tau =  \prod_{\tau = 1}^t (1 - \beta_t),\ \beta_t\in [\beta_{\mathrm{start}},\beta_{\mathrm{end}}]$ pedig a zaj-inkremensek (lineáris felfutású) ütemezése. 

A modell a zajt jósolja:

$$
\varepsilon_\theta(x_t, t)
$$


A tanításhoz használt loss:

$$
\mathcal{L} = \mathbb{E} \|\varepsilon_\theta(x_t, t) - \varepsilon\|^2
$$


In [ ]:

# 1.1 Adat: Two Moons
data_np, _ = make_moons(n_samples=cfg.n_points_2d, noise=0.06, random_state=42)
data_np = (data_np - data_np.mean(axis=0)) / data_np.std(axis=0)

dataset_2d = TensorDataset(torch.tensor(data_np, dtype=torch.float32))
loader_2d = DataLoader(
    dataset_2d,
    batch_size=cfg.batch_size_2d,
    shuffle=True,
    drop_last=True,
)

fig, ax = plt.subplots(figsize=(5, 5))
show_2d_points(data_np, title="Two Moons adathalmaz", ax=ax)
plt.show()

In [ ]:
# 1.2 Diffúziós konstansok
T = 1000
beta_start = 1e-4
beta_end = 0.02

# Ugyanazt a mintát követjük, mint a korábbi notebookokban:
# minden fontos tensor ugyanarra a globális device-ra kerül.
betas = torch.linspace(beta_start, beta_end, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

def extract(a, t, x_shape):
    """Kiválasztja az a[t] elemeket, majd broadcasting-kompatibilis alakra hozza.

    Feltételezzük, hogy `a` és `t` ugyanazon az eszközön vannak.
    Ez illeszkedik a korábbi E04 notebookok egyszerű, globális `device` stílusához.
    """
    batch_size = t.shape[0]
    out = a[t]
    return out.view(batch_size, *([1] * (len(x_shape) - 1)))

print("T =", T)
print("beta tartomány:", betas[0].item(), "→", betas[-1].item())
print("alpha_bar(T) =", alphas_cumprod[-1].item())


## Miért kell az `extract()`?
A diffúziós mennyiségek (`betas`, `sqrt_alphas_cumprod`, stb.) 1D vektorok, egy-egy
időlépéshez tartozó konstansokat tárolnak.

Csakhogy egy batch-en belül **nem biztos, hogy minden elemhez ugyanaz a `t` tartozik**.
Ezért kell mintánként kiválasztani a megfelelő értéket, majd olyan alakra hozni,
hogy broadcastinggal megszorozhassuk az egész képet vagy pontot.

**Emlékeztető:**
- DDPM-ben `t` egész index (`long`), pl. `t = [0, 10, 100, 999]`
- FM-ben `t` lebegőpontos idő (`float`), pl. `t = [0.1, 0.4, 0.8]`


In [ ]:
# Mini példa: minden mintához más időindex tartozik
alpha_demo = torch.linspace(0.1, 0.8, 8, device=device)
t_index_demo = torch.tensor([0, 5, 2, 7], dtype=torch.long, device=device)

alpha_t = alpha_demo[t_index_demo]          # shape: (B,)
alpha_t_img = alpha_t.view(4, 1, 1, 1)      # shape: (B,1,1,1)

x_img_demo = torch.randn(4, 1, 28, 28, device=device)

print_shape("alpha_demo", alpha_demo)
print_shape("t_index_demo", t_index_demo)
print_shape("alpha_t", alpha_t)
print_shape("alpha_t_img", alpha_t_img)
print_shape("alpha_t_img * x_img_demo", alpha_t_img * x_img_demo)


In [ ]:

# 1.3 Modell: egyszerű MLP zaj-előrejelzésre

class SinusoidalEmbedding(nn.Module):
    """Szinuszos időbeágyazás.

    Bemenet:  t.shape == (B,)
    Kimenet: emb.shape == (B, dim)
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / half
        )
        args = t.float()[:, None] * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class NoisePredictor(nn.Module):
    """MLP, amely a hozzáadott zajt próbálja megjósolni."""
    def __init__(self, data_dim=2, hidden=128, time_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalEmbedding(time_dim),
            nn.Linear(time_dim, hidden),
            nn.SiLU(),
        )
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x, t):
        t_emb = self.time_mlp(t)       # (B, hidden)
        h = torch.cat([x, t_emb], dim=-1)
        return self.net(h)             # (B, 2)

model_2d = NoisePredictor().to(device)
print("Paraméterszám:", sum(p.numel() for p in model_2d.parameters()))

## 1.4 TODO – `q_sample`

Ez a lépés szerkezetileg nagyon hasonlít a korábbi denoiser példára:
most is létrehozunk egy zajos bemenetet, csak most **mi magunk építjük fel**
az `x_t` állapotot az `x_0` és a zaj lineáris kombinációjaként.


In [ ]:
def q_sample(x_start, t, noise=None):
    """Forward diffusion: x0 -> xt."""
    if noise is None:
        noise = torch.randn_like(x_start)

    # TODO:
    # 1. Kérd le a sqrt(alpha_bar_t) értékeket.
    # 2. Kérd le a sqrt(1 - alpha_bar_t) értékeket.
    # 3. Számold ki xt-t a képlet alapján.
    # 4. Térj vissza xt-vel.

    raise NotImplementedError("Implementáld a q_sample függvényt!")

In [ ]:
# Sanity check q_sample-hez
try:
    _x0 = torch.randn(4, 2, device=device)
    _t = torch.tensor([0, 10, 100, 999], dtype=torch.long, device=device)
    _xt = q_sample(_x0, _t)
    print_shape("x0", _x0)
    print_shape("xt", _xt)
    assert _xt.shape == _x0.shape
    print("q_sample: shape rendben.")
except NotImplementedError as e:
    print(e)


In [ ]:
# Forward folyamat vizualizálása
try:
    timesteps_vis = [0, 100, 300, 600, 999]
    x0_vis = torch.tensor(data_np[:1000], dtype=torch.float32, device=device)
    fig, axes = plt.subplots(1, len(timesteps_vis), figsize=(15, 3))
    for ax, t_idx in zip(axes, timesteps_vis):
        t_batch = torch.full((len(x0_vis),), t_idx, dtype=torch.long, device=device)
        xt = q_sample(x0_vis, t_batch)
        show_2d_points(xt, title=f"t={t_idx}", ax=ax)
        ax.set_xlim(-3.5, 3.5)
        ax.set_ylim(-3.5, 3.5)
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Előbb implementáld a q_sample függvényt.")


## 1.5 TODO – `p_losses`

A PyTorch-minta itt **pont ugyanaz**, mint a korábbi denoiser notebookban:
1. előállítasz egy bemenetet,
2. meghívod a modellt,
3. kiszámolod az MSE loss-t.

Az újdonság csak annyi, hogy a target most a **valóban hozzáadott zaj**.


In [ ]:
def p_losses(model, x_start, t):
    """DDPM tanítási veszteség egy batch-re."""
    # TODO:
    # 1. Generálj zajt: noise = torch.randn_like(x_start)
    # 2. Számold ki xt-t q_sample-lel.
    # 3. predicted_noise = model(xt, t)
    # 4. loss = mse_criterion(predicted_noise, noise)
    # 5. Térj vissza a loss-szal.

    raise NotImplementedError("Implementáld a p_losses függvényt!")

In [ ]:
# Sanity check p_losses-hez
try:
    _model = NoisePredictor().to(device).to(device)
    _x = torch.randn(8, 2, device=device)
    _t = torch.randint(0, T, (8,), dtype=torch.long, device=device)
    _loss = p_losses(_model, _x, _t)
    print("loss =", float(_loss))
    assert _loss.ndim == 0
    print("p_losses: OK")
except NotImplementedError as e:
    print(e)


In [ ]:

# 1.6 DDPM tanítás
optimizer_2d = optim.Adam(model_2d.parameters(), lr=1e-3)
losses_2d = []

try:
    for epoch in tqdm(range(cfg.epochs_ddpm_2d), desc="DDPM tanítás (2D)"):
        epoch_loss = 0.0
        for (batch,) in loader_2d:
            batch = batch.to(device)
            t = torch.randint(0, T, (batch.shape[0],), device=device)

            loss = p_losses(model_2d, batch, t)

            optimizer_2d.zero_grad()
            loss.backward()
            optimizer_2d.step()

            epoch_loss += loss.item()

        losses_2d.append(epoch_loss / len(loader_2d))

    plt.figure(figsize=(7, 3))
    plt.plot(losses_2d)
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title("DDPM tanítási görbe (2D)")
    plt.yscale("log")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Előbb implementáld a p_losses függvényt.")

## 1.7 TODO – `ddpm_sample_step`

Használandó képlet:

$$
\mu_t = \frac{1}{\sqrt{\alpha_t}}
\left(
x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}} \varepsilon_\theta(x_t,t)
\right)
$$


majd

$$
x_{t-1} = \mu_t + \sqrt{\tilde\beta_t} z
$$

ahol $z \sim N(0, I)$ ha `t_idx > 0`, különben nincs zaj.

**Tipp:** minden konstansra használd az `extract()` függvényt.

In [ ]:

@torch.no_grad()
def ddpm_sample_step(model, x, t_idx):
    """Egy DDPM visszalépés: xt -> x_{t-1}."""
    B = x.shape[0]
    t = torch.full((B,), t_idx, device=x.device, dtype=torch.long)

    # TODO:
    # 1. beta_t = extract(betas, t, x.shape)
    # 2. sqrt_one_minus_ab = extract(sqrt_one_minus_alphas_cumprod, t, x.shape)
    # 3. sqrt_recip_alpha_t = extract(sqrt_recip_alphas, t, x.shape)
    # 4. predicted_noise = model(x, t)
    # 5. model_mean = ...
    # 6. Ha t_idx == 0: return model_mean
    # 7. Különben adj hozzá posterior_variance alapján zajt.

    raise NotImplementedError("Implementáld a ddpm_sample_step függvényt!")

In [ ]:
# Sanity check ddpm_sample_step-hez
try:
    _model = NoisePredictor().to(device).to(device)
    _x = torch.randn(16, 2, device=device)
    _out = ddpm_sample_step(_model, _x, t_idx=10)
    print_shape("x_in", _x)
    print_shape("x_out", _out)
    assert _out.shape == _x.shape
    print("ddpm_sample_step: shape rendben.")
except NotImplementedError as e:
    print(e)


In [ ]:

# DDPM mintavételezés és trajektóriák
try:
    model_2d.eval()

    n_samples_vis = 1500
    n_traj = 150
    save_every = 25

    x = torch.randn(n_samples_vis, 2, device=device)
    traj = [x[:n_traj].cpu().clone()]

    for t_idx in tqdm(reversed(range(T)), total=T, desc="DDPM mintavétel"):
        x = ddpm_sample_step(model_2d, x, t_idx)
        if t_idx % save_every == 0:
            traj.append(x[:n_traj].cpu().clone())

    traj = torch.stack(traj)
    samples_ddpm = x.cpu()

    fig, ax = plt.subplots(figsize=(6, 6))
    for i in range(n_traj):
        ax.plot(traj[:, i, 0], traj[:, i, 1], c="red", alpha=0.12, linewidth=0.7)
    ax.scatter(traj[0, :, 0], traj[0, :, 1], s=6, alpha=0.4, c="red", label="kezdő zaj")
    ax.scatter(samples_ddpm[:, 0], samples_ddpm[:, 1], s=3, alpha=0.5, c="C0", label="végpont")
    ax.set_title("DDPM minták")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect("equal")
    ax.legend()
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Előbb implementáld a ddpm_sample_step függvényt.")

## 1.8 Opcionális bónusz – `ddim_sample_step`

Ez a rész **nem szükséges a minimum beadáshoz**, de erősen ajánlott annak, aki már kész a DDPM-es alaplépéssel.

A determinisztikus DDIM lépés:
1. becsüld meg `x0_hat`-et
2. ebből számold ki `x_{t_prev}`-et

$$
\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\varepsilon_\theta}{\sqrt{\bar\alpha_t}}
$$

$$
x_{t_{prev}} = \sqrt{\bar\alpha_{t_{prev}}}\,\hat{x}_0 +
\sqrt{1-\bar\alpha_{t_{prev}}}\,\varepsilon_\theta
$$


In [ ]:

@torch.no_grad()
def ddim_sample_step(model, x, t_idx, t_prev_idx):
    """Egy DDIM lépés: x_t -> x_{t_prev}."""
    B = x.shape[0]
    t = torch.full((B,), t_idx, device=x.device, dtype=torch.long)

    # TODO:
    # 1. Kérd le a szükséges sqrt(alpha_bar_t) és sqrt(1-alpha_bar_t) konstansokat.
    # 2. predicted_noise = model(x, t)
    # 3. x0_hat = ...
    # 4. Ha t_prev_idx < 0: return x0_hat
    # 5. Egyébként számold ki x_prev-et a DDIM képletből.

    raise NotImplementedError("Implementáld a ddim_sample_step függvényt!")

In [ ]:
# DDIM sanity check
try:
    _model = NoisePredictor().to(device).to(device)
    _x = torch.randn(16, 2, device=device)
    _out = ddim_sample_step(_model, _x, t_idx=100, t_prev_idx=80)
    print_shape("x_in", _x)
    print_shape("x_out", _out)
    assert _out.shape == _x.shape
    print("ddim_sample_step: shape rendben.")
except NotImplementedError as e:
    print(e)


In [ ]:

# DDIM mintavételezés különböző lépésszámmal
try:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    n_traj = 120

    for ax, n_steps in zip(axes, [20, 50, 200]):
        x = torch.randn(1500, 2, device=device)
        traj = [x[:n_traj].cpu().clone()]

        step_indices = list(np.linspace(0, T - 1, n_steps, dtype=int))
        step_indices = sorted(set(step_indices))
        pairs = list(zip(reversed(step_indices), list(reversed(step_indices))[1:] + [-1]))

        for t_idx, t_prev_idx in pairs:
            x = ddim_sample_step(model_2d, x, int(t_idx), int(t_prev_idx))
            traj.append(x[:n_traj].cpu().clone())

        traj = torch.stack(traj)
        for i in range(n_traj):
            ax.plot(traj[:, i, 0], traj[:, i, 1], c="red", alpha=0.12, linewidth=0.7)
        ax.scatter(traj[0, :, 0], traj[0, :, 1], s=5, alpha=0.35, c="red")
        ax.scatter(x.cpu()[:, 0], x.cpu()[:, 1], s=3, alpha=0.5)
        ax.set_title(f"DDIM – {n_steps} lépés")
        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)
        ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Előbb implementáld a ddim_sample_step függvényt.")

## 1.9 Eredmények értékelése
Futtasd le a két módszert, és válaszolj röviden:
1. Hogyan minősítenéd a kapott eredményt? 

2. Mit gondolsz, hogyan lehetne jobb eredményeket produkálni? Kísérletezz!

3. (Ha megcsináltad a bőnuszt: miben különbözik a DDPM és a DDIM trajektóriája?)


---

# 1.10 2D Flow Matching

Itt nem zajt távolítunk el sok apró lépésben, hanem egy **sebességmezőt** tanulunk, amely
folyamatosan viszi a pontokat a forrás-eloszlásból a cél-eloszlásba.

## Képlet

$$
x_t = (1-t)x_0 + t x_1
$$


$$
v_{target} = x_1 - x_0
$$


In [ ]:
# Forrás és cél pontfelhő
N_SPIRAL = cfg.n_points_2d
theta_spiral = np.linspace(0, 3 * np.pi, N_SPIRAL)
r_spiral = theta_spiral

spiral_np = np.column_stack([
    r_spiral * np.cos(theta_spiral),
    r_spiral * np.sin(theta_spiral)
])
spiral_np += np.random.RandomState(42).randn(N_SPIRAL, 2) * 0.15
spiral_np = (spiral_np - spiral_np.mean(axis=0)) / spiral_np.std(axis=0)

OFFSET_SRC = np.array([-2.0, 0.0])
OFFSET_TGT = np.array([ 2.0, 0.0])

spiral_np = spiral_np + OFFSET_SRC
target_fm_np = data_np + OFFSET_TGT

# A 2D flow matching részben egyszerűbb, ha a tensorok eleve ugyanazon az eszközön vannak,
# mint a modell. Így a TODO-kban nem kell külön device-kezeléssel foglalkozni.
source_tensor = torch.tensor(spiral_np, dtype=torch.float32, device=device)
target_tensor = torch.tensor(target_fm_np, dtype=torch.float32, device=device)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
show_2d_points(spiral_np, title="Forrás: spirál", ax=axes[0], color="C3")
show_2d_points(target_fm_np, title="Cél: two moons", ax=axes[1], color="C0")
axes[0].set_xlim(-5, 5); axes[0].set_ylim(-4, 4)
axes[1].set_xlim(-5, 5); axes[1].set_ylim(-4, 4)
plt.tight_layout()
plt.show()


In [ ]:

class VelocityPredictor(nn.Module):
    """MLP, amely v(x_t, t) sebességmezőt becsül."""
    def __init__(self, data_dim=2, hidden=128, time_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalEmbedding(time_dim),
            nn.Linear(time_dim, hidden),
            nn.SiLU(),
        )
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x, t):
        t_emb = self.time_mlp(t * 1000)  # segít a beágyazásnak
        h = torch.cat([x, t_emb], dim=-1)
        return self.net(h)

vel_model = VelocityPredictor().to(device).to(device)
print("VelocityPredictor paraméterszám:", sum(p.numel() for p in vel_model.parameters()))

## 1.10/a TODO – `fm_2d_train_step`

Itt most szándékosan **egyszerű, kifejtett kódot** szeretnénk.

Lépések:
1. válassz ki egy batch-nyi forráspontot és célpontot
2. mintázz $t \sim U(0,1)$ időket
3. állítsd elő $x_t$-t lineáris interpolációval
4. számold ki a célszebességet: $x_1 - x_0$
5. MSE loss + backward + optimizer step

Nem kell tömör megoldás. A több, de jól olvasható sor teljesen rendben van.


In [ ]:
def fm_2d_train_step(model, optimizer, source, target, batch_size=256):
    """Egy 2D flow matching tanítási lépés."""
    B = batch_size

    # TODO:
    # 1. Válassz ki B darab forráspontot és B darab célpontot.
    #       idx0 = torch.randint(0, source.shape[0], (B,), device=device)
    #       idx1 = torch.randint(0, target.shape[0], (B,), device=device)
    #       x0 = source[idx0]
    #       x1 = target[idx1]
    # 2. Mintázz időket: t = torch.rand(B, device=device)
    # 3. Broadcastinghoz: t_expand = t.view(B, 1)
    # 4. Interpoláció: x_t = (1 - t_expand) * x0 + t_expand * x1
    # 5. Célsebesség: v_target = ...
    # 6. Modellhívás: pred_v = ...
    # 7. Loss: loss = mse_criterion(pred_v, v_target)
    # 8. optimizer léptetése
    # 9. return loss.detach()

    raise NotImplementedError("Implementáld az fm_2d_train_step függvényt!")


In [ ]:
# fm_2d_train_step sanity check
try:
    _model = VelocityPredictor().to(device).to(device)
    _opt = optim.Adam(_model.parameters(), lr=1e-3)
    _loss = fm_2d_train_step(_model, _opt, source_tensor, target_tensor, batch_size=32)
    print("loss =", float(_loss))
    print("fm_2d_train_step: OK")
except NotImplementedError as e:
    print(e)


In [ ]:

# 2D Flow Matching tanítás
vel_optimizer = optim.Adam(vel_model.parameters(), lr=1e-3)
fm2d_losses = []

try:
    for epoch in tqdm(range(cfg.epochs_fm_2d), desc="2D FM tanítás"):
        epoch_loss = 0.0
        for _ in range(cfg.steps_per_epoch_fm2d):
            loss = fm_2d_train_step(
                vel_model, vel_optimizer, source_tensor, target_tensor,
                batch_size=cfg.batch_size_2d
            )
            epoch_loss += float(loss)
        fm2d_losses.append(epoch_loss / cfg.steps_per_epoch_fm2d)

    plt.figure(figsize=(7, 3))
    plt.plot(fm2d_losses)
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title("2D Flow Matching tanítási görbe")
    plt.yscale("log")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Előbb implementáld az fm_2d_train_step függvényt.")

## 1.10/b `fm_2d_sample`

Ez a sampler ugyanazt az ötletet követi, mint később az MNIST-es Euler mintavételezés,
csak itt 2D pontokkal dolgozunk.


In [ ]:
@torch.no_grad()
def fm_2d_sample(model, source_points, n_steps=100, record_every=None):
    """2D flow matching mintavételezés Euler-integrációval.

    Egyszerű, tanulóbarát változat:
    - a source_points már a megfelelő eszközön van,
    - a függvény ugyanazon az eszközön adja vissza az eredményt,
    - rajzolás előtt majd külön hívunk `.cpu()`-t.
    """
    x = source_points.clone()
    dt = 1.0 / n_steps

    if record_every is not None:
        traj = [x.clone()]

    B = x.shape[0]

    for i in range(n_steps):
        t = torch.full((B,), i / n_steps, device=x.device)
        v = model(x, t)
        x = x + dt * v

        if record_every is not None:
            if (i + 1) % record_every == 0 or i == n_steps - 1:
                traj.append(x.clone())

    if record_every is not None:
        traj = torch.stack(traj)
        return x, traj

    return x


In [ ]:

# fm_2d_sample sanity check
try:
    _src = source_tensor[:20]
    _out = fm_2d_sample(vel_model, _src, n_steps=10)
    if isinstance(_out, tuple):
        _out = _out[0]
    print_shape("source", _src)
    print_shape("output", _out)
    assert _out.shape == _src.shape
    print("fm_2d_sample: shape rendben.")
except NotImplementedError as e:
    print(e)

In [ ]:
# 2D FM mintavételezés és trajektóriák
try:
    n_vis = 200
    idx_vis = np.random.choice(len(spiral_np), n_vis, replace=False)
    source_vis = source_tensor[idx_vis]

    samples_fm2d = fm_2d_sample(vel_model, source_tensor, n_steps=100)
    _, traj = fm_2d_sample(vel_model, source_vis, n_steps=100, record_every=5)

    if isinstance(samples_fm2d, tuple):
        samples_fm2d = samples_fm2d[0]

    # A rajzoláshoz CPU-ra visszük a tensorokat.
    samples_fm2d = samples_fm2d.cpu()
    traj = traj.cpu()

    fig, ax = plt.subplots(figsize=(8, 6))
    for i in range(n_vis):
        ax.plot(traj[:, i, 0], traj[:, i, 1], c="red", alpha=0.2, linewidth=0.7)
    ax.scatter(traj[0, :, 0], traj[0, :, 1], s=8, alpha=0.6, c="C3", label="forrás")
    ax.scatter(samples_fm2d[:, 0], samples_fm2d[:, 1], s=8, alpha=0.6, c="C0", label="FM kimenet")
    ax.set_title("2D Flow Matching")
    ax.set_xlim(-5, 5); ax.set_ylim(-4, 4); ax.set_aspect("equal")
    ax.legend()
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Előbb implementáld az fm_2d_sample függvényt.")


---

# 2. feladat – Feltételes Flow Matching MNIST-en

Ez a rész már összetettebb, de a fő PyTorch-minta ugyanaz marad, mint a korábbi MNIST
osztályozó és denoiser notebookokban.

## Kapcsolat a korábbi példákkal

| Korábban | Most |
|---|---|
| input: kép | input: interpolált kép `x_t` |
| extra input: nincs | extra input: `t` és címke |
| target: osztály vagy tiszta kép | target: `v_target = x_1 - x_0` |
| loss: CrossEntropy vagy MSE | loss: MSE |

## Mit **nem** kell implementálnod?
A `CondFlowUNet.forward(...)` metódust tekintsd fekete doboznak, amelynek interfésze:

```python
model(x_t, t, labels) -> prediction
```

A fókusz itt ezen van:
1. hogyan készül az `x_t`,
2. mi a `v_target`,
3. hogyan történik a tanítás,
4. hogyan működik a mintavételezés,
5. és hogyan módosítja ezt a CFG.


In [ ]:

# 2.1 MNIST adatbetöltés
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),   # [0,1] -> [-1,1]
])

mnist_train_full = torchvision.datasets.MNIST(
    "./data", train=True, download=True, transform=mnist_transform
)
mnist_test = torchvision.datasets.MNIST(
    "./data", train=False, download=True, transform=mnist_transform
)

if cfg.mnist_train_subset < len(mnist_train_full):
    indices = list(range(cfg.mnist_train_subset))
    mnist_train = Subset(mnist_train_full, indices)
else:
    mnist_train = mnist_train_full

loader_mnist = DataLoader(
    mnist_train,
    batch_size=cfg.batch_size_mnist,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    mnist_test,
    batch_size=cfg.batch_size_mnist,
    shuffle=False,
    num_workers=0,
)

fig, axes = plt.subplots(1, 10, figsize=(12, 1.5))
for i in range(10):
    img, lbl = mnist_train_full[i]
    axes[i].imshow(img.squeeze(), cmap="gray")
    axes[i].axis("off")
    axes[i].set_title(str(lbl), fontsize=9)
plt.suptitle("MNIST minták")
plt.tight_layout()
plt.show()

## Shape-emlékeztető MNIST-hez
- `x1`: `(B, 1, 28, 28)` — célkép
- `x0`: `(B, 1, 28, 28)` — zaj
- `t`: `(B,)`
- `t_expand`: `(B, 1, 1, 1)`
- `labels`: `(B,)`
- `v_target`: `(B, 1, 28, 28)`

In [ ]:
class SinusEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / half
        )
        args = t.float()[:, None] * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ResBlock(nn.Module):
    """Egyszerű residual blokk idő+osztály beágyazással."""
    def __init__(self, in_ch, out_ch, emb_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        # A beágyazást channel-dimenzió mentén adjuk hozzá
        h = h + self.emb_proj(emb)[:, :, None, None]

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)

        return h + self.skip(x)

class CondFlowUNet(nn.Module):
    """Kis U-Net flow matchinghez, osztálykondicionálással."""
    def __init__(self, in_ch=1, base=32, emb_dim=128, num_classes=10):
        super().__init__()
        self.num_classes = num_classes

        self.time_emb = nn.Sequential(
            SinusEmbedding(emb_dim),
            nn.Linear(emb_dim, emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim),
        )
        self.class_emb = nn.Embedding(num_classes + 1, emb_dim)  # +1 = null class

        # Encoder: 28x28 -> 14x14 -> 7x7
        self.enc1 = ResBlock(in_ch, base, emb_dim)
        self.enc2 = ResBlock(base, base * 2, emb_dim)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.mid = ResBlock(base * 2, base * 4, emb_dim)

        # Decoder
        self.up2 = nn.ConvTranspose2d(base * 4, base * 4, kernel_size=2, stride=2)
        self.dec2 = ResBlock(base * 4 + base * 2, base * 2, emb_dim)

        self.up1 = nn.ConvTranspose2d(base * 2, base * 2, kernel_size=2, stride=2)
        self.dec1 = ResBlock(base * 2 + base, base, emb_dim)

        self.out_conv = nn.Conv2d(base, in_ch, kernel_size=1)

    def forward(self, x, t, labels=None):
        """
        Bemenet:
            x:      (B, 1, 28, 28)
            t:      (B,)
            labels: (B,) vagy None
        Kimenet:
            v:      (B, 1, 28, 28)
        """
        B = x.shape[0]

        time_embedding = self.time_emb(t)

        if labels is None:
            labels = torch.full((B,), self.num_classes, device=x.device, dtype=torch.long)

        class_embedding = self.class_emb(labels)
        emb = time_embedding + class_embedding

        h1 = self.enc1(x, emb)
        h2 = self.enc2(self.pool(h1), emb)
        hmid = self.mid(self.pool(h2), emb)

        h = self.up2(hmid)
        h = torch.cat([h, h2], dim=1)
        h = self.dec2(h, emb)

        h = self.up1(h)
        h = torch.cat([h, h1], dim=1)
        h = self.dec1(h, emb)

        out = self.out_conv(h)
        return out

flow_model = CondFlowUNet(in_ch=1, base=32, emb_dim=128, num_classes=10).to(device)
print("CondFlowUNet paraméterszám:", sum(p.numel() for p in flow_model.parameters()))


In [ ]:

# CondFlowUNet sanity check
try:
    _x = torch.randn(2, 1, 28, 28, device=device)
    _t = torch.rand(2, device=device)
    _labels = torch.tensor([3, 7], device=device)
    _out = flow_model(_x, _t, _labels)
    print_shape("input", _x)
    print_shape("output", _out)
    assert _out.shape == _x.shape
    print("CondFlowUNet.forward: OK")
except NotImplementedError as e:
    print(e)

In [ ]:
def apply_label_dropout(labels, p_uncond, null_class):
    """Classifier-free guidance tanításhoz véletlenül null class-ra cserél címkéket."""
    labels_cfg = labels.clone()
    drop_mask = torch.rand(labels.shape[0], device=labels.device) < p_uncond
    labels_cfg[drop_mask] = null_class
    return labels_cfg


## 2.3 TODO – `fm_train_step`

Itt ugyanaz a séma, mint a korábbi tanítási hurkokban:
1. előkészíted a bemenetet és a targetet,
2. meghívod a modellt,
3. kiszámolod az MSE loss-t,
4. `zero_grad()`, `backward()`, `step()`.

A CFG-hez szükséges label dropoutot külön helper függvény kezeli.


In [ ]:
def fm_train_step(model, optimizer, x1, labels, p_uncond=0.1):
    """Egy flow matching tanítási lépés MNIST-en."""
    B = x1.shape[0]

    # TODO:
    # 1. Zajminta: x0 = ...
    # 2. Véletlen idők: t = ...
    # 3. Broadcastinghoz: t_expand = t.view(B, 1, 1, 1)
    # 4. Interpoláció: x_t = ...
    # 5. Célszebesség: v_target = ...
    # 6. Label dropout helper: labels_cfg = apply_label_dropout(labels, p_uncond, model.num_classes)
    # 7. Modellhívás: pred_v = ...
    # 8. Loss: loss = mse_criterion(pred_v, v_target)
    # 9. optimizer lépttetése
    # 10. return loss.detach()

    raise NotImplementedError("Implementáld az fm_train_step függvényt!")


In [ ]:

# fm_train_step sanity check
try:
    _images = torch.randn(4, 1, 28, 28, device=device)
    _labels = torch.tensor([1, 2, 3, 4], device=device)
    _opt = optim.Adam(flow_model.parameters(), lr=1e-3)
    _loss = fm_train_step(flow_model, _opt, _images, _labels, p_uncond=0.1)
    print("loss =", float(_loss))
    print("fm_train_step: OK")
except NotImplementedError as e:
    print(e)

In [ ]:

# Rövid smoke test: 1-2 batch
try:
    _test_opt = optim.Adam(flow_model.parameters(), lr=1e-3)
    flow_model.train()
    for i, (images, labels) in enumerate(loader_mnist):
        images = images.to(device)
        labels = labels.to(device)
        loss = fm_train_step(flow_model, _test_opt, images, labels, p_uncond=0.1)
        print(f"batch {i}: loss = {float(loss):.4f}")
        if i == 1:
            break
except NotImplementedError:
    print("Előbb implementáld az fm_train_step függvényt és a modell forwardját.")

In [ ]:

# Teljesebb tanítás
fm_optimizer = optim.Adam(flow_model.parameters(), lr=1e-3)
fm_losses = []

try:
    for epoch in range(1, cfg.epochs_mnist + 1):
        flow_model.train()
        epoch_loss = 0.0
        n_batches = 0

        pbar = tqdm(loader_mnist, desc=f"FM Epoch {epoch}/{cfg.epochs_mnist}")
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            loss = fm_train_step(flow_model, fm_optimizer, images, labels, p_uncond=0.1)
            epoch_loss += float(loss)
            n_batches += 1

            pbar.set_postfix({"loss": f"{float(loss):.4f}"})

        avg = epoch_loss / n_batches
        fm_losses.append(avg)
        print(f"Epoch {epoch}: átlagos loss = {avg:.4f}")

    plt.figure(figsize=(7, 3))
    plt.plot(fm_losses)
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title("Flow Matching tanítási görbe (MNIST)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Előbb implementáld a hiányzó MNIST-es részeket.")

## 2.4 TODO – `fm_euler_sample`

Alapul veheted 1.10/b-t. 

Mintavételezés:
1. indulj Gauss-zajból,
2. minden lépésben kérd le `v = model(x, t, labels)`,
3. Euler-lépés: `x = x + dt * v`.

Ugyanaz a globális `device` sémánk, mint a korábbi notebookokban.


In [ ]:
@torch.no_grad()
def fm_euler_sample(model, labels, num_steps=100):
    """Flow matching mintavételezés Euler-integrációval."""

    # TODO:
    # 1. labels = labels.to(device)
    # 2. Címkék száma - N = ...
    # 3. Random kép  - x = ...
    # 4. dt = 1.0 / num_steps
    # 5. for i in range(num_steps):
    #       t = torch.full((N,), i / num_steps, device=device)
    #       v = ...
    #       x = ...
    # 6. return x

    raise NotImplementedError("Implementáld az fm_euler_sample függvényt!")


In [ ]:

# fm_euler_sample sanity check
try:
    _labels = torch.arange(10, device=device)
    _imgs = fm_euler_sample(flow_model, _labels, num_steps=10)
    print_shape("generated", _imgs)
    assert _imgs.shape == (10, 1, 28, 28)
    print("fm_euler_sample: OK")
except NotImplementedError as e:
    print(e)

In [ ]:

# Minták Euler-integrációval
try:
    labels_gen = torch.arange(10, device=device).repeat_interleave(5)
    gen_imgs = fm_euler_sample(flow_model, labels_gen, num_steps=100).cpu()

    fig, axes = plt.subplots(5, 10, figsize=(12, 6))
    for i in range(50):
        row, col = divmod(i, 10)
        img = (gen_imgs[i].squeeze() + 1) / 2
        axes[row, col].imshow(img.clamp(0, 1), cmap="gray")
        axes[row, col].axis("off")
    for c in range(10):
        axes[0, c].set_title(str(c), fontsize=9)
    plt.suptitle("Euler mintavételezés")
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Előbb implementáld az fm_euler_sample függvényt.")

## 2.5 Opcionális bónusz – `fm_cfg_sample`

Ez a rész **nem szükséges a minimum beadáshoz**, de jó kiegészítés, ha már működik az alap flow matching mintavételezés.

Classifier-Free Guidance esetén ugyanaz az Euler-ciklus marad, csak minden lépésben
**két predikciót** számolunk:
1. feltétel nélküli `v_uncond` (null címke feltétel),
2. feltételes `v_cond`,
3. majd ezeket kombináljuk:

$$
v_{cfg} = v_{uncond} + w (v_{cond} - v_{uncond})
$$


In [ ]:
@torch.no_grad()
def fm_cfg_sample(model, labels, guidance_scale=3.0, num_steps=100):
    """Mintavételezés classifier-free guidance-szel."""

    # TODO:
    # 1. labels = labels.to(device)
    # 2. Címkék száma - N = labels.shape[0]
    # 3. Random kép  - x = torch.randn(N, 1, 28, 28, device=device)
    # 4. null_labels = torch.full((N,), model.num_classes, device=device, dtype=torch.long)
    # 5. dt = 1.0 / num_steps
    # 6. for i in range(num_steps):
    #       t = torch.full((N,), i / num_steps, device=device)
    #       v_uncond = model(x, t, null_labels)
    #       v_cond = model(x, t, labels)
    #       v = v_uncond + guidance_scale * (v_cond - v_uncond)
    #       x = x + dt * v
    # 7. return x

    raise NotImplementedError("Implementáld az fm_cfg_sample függvényt!")


In [ ]:

# fm_cfg_sample sanity check
try:
    _labels = torch.arange(10, device=device)
    _imgs = fm_cfg_sample(flow_model, _labels, guidance_scale=3.0, num_steps=10)
    print_shape("cfg_generated", _imgs)
    assert _imgs.shape == (10, 1, 28, 28)
    print("fm_cfg_sample: OK")
except NotImplementedError as e:
    print(e)

In [ ]:

# CFG hatásának vizualizációja
try:
    fig, axes_rows = plt.subplots(4, 10, figsize=(12, 5))
    target_labels = torch.arange(10, device=device)

    for row, w in enumerate([0.0, 1.0, 3.0, 7.0]):
        if w == 0.0:
            null_labels = torch.full((10,), flow_model.num_classes, device=device, dtype=torch.long)
            imgs = fm_euler_sample(flow_model, null_labels, num_steps=100).cpu()
        else:
            imgs = fm_cfg_sample(
                flow_model,
                target_labels,
                guidance_scale=w,
                num_steps=100,
            ).cpu()

        for col in range(10):
            img = (imgs[col].squeeze() + 1) / 2
            axes_rows[row, col].imshow(img.clamp(0, 1), cmap="gray")
            axes_rows[row, col].axis("off")
        axes_rows[row, 0].set_ylabel(f"w={w:.0f}", fontsize=9, rotation=0, labelpad=25)

    for c in range(10):
        axes_rows[0, c].set_title(str(c), fontsize=9)

    plt.suptitle("Classifier-Free Guidance hatása")
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Előbb implementáld az fm_cfg_sample függvényt.")

---

# Beadandó

## Minimum beadási szint
A notebook akkor tekinthető késznek, ha:

1. a **kötelező** TODO-k implementálva vannak,
2. a kötelező részek sanity checkjei sikeresen lefutnak,
3. a fő vizualizációk megjelennek,
4. a notebook a kötelező részekig elejétől a végéig futtatható.

## Opcionális bónusz
Ha marad időd, implementáld még:
- `ddim_sample_step`
- `fm_cfg_sample`

## Mit érdemes átgondolni a feladat kapcsán?
- DDPM és flow matching alapötletének különbsége
- mi a szerepe az `x_t` zajos / interpolált állapotnak
- miért kell null class a CFG-hez
- milyen hatása volt a guidance scale-nek (ha megcsináltad a bónuszt)

## Tipikus hibák
- rossz `shape`
- DDPM-ben `t` nem `long`
- FM-ben elmarad a `t.view(...)` broadcastinghoz
- a batch nincs ugyanazon a `device`-on, mint a modell
- `model.eval()` hiánya mintavételezéskor
- `torch.no_grad()` hiánya samplingnél
